# CS164 — Session 16: Linear Programming (Python / CVXPY)

## Q7 — Numerical verification

This notebook solves the industrial problem from Q5 in matrix form:

\[
\max_z c^Tz\quad\text{subject to }Az\leq b,\ z\geq0
\]

\[
c=\begin{bmatrix}8\\6\end{bmatrix},\qquad
A=\begin{bmatrix}1&1\\5&10\\2&1\end{bmatrix},\qquad
b=\begin{bmatrix}2\\16\\3\end{bmatrix}.
\]

Switch to the Python kernel. Install dependencies with \`%pip install -r requirements.txt\` from the repository root (or \`%pip install numpy cvxpy\`).

Expected optimum: \(z^*=(1,1)\), profit **14**, and \(Az^*=(2,15,3)\leq(2,16,3)\).

The companion [R notebook](16_linear_programming_R.ipynb) covers Q1–6 and Q8.

In [ ]:
"""CS164 Session 16: Question 7, industrial LP using Python and CVXPY."""
import numpy as np
import cvxpy as cp

def solve_industrial():
    c = np.array([8., 6.])
    A = np.array([[1., 1.],
                  [5., 10.],
                  [2., 1.]])
    b = np.array([2., 16., 3.])
    x = cp.Variable(2, nonneg=True)
    problem = cp.Problem(cp.Maximize(c @ x), [A @ x <= b])
    problem.solve()
    if problem.status not in ("optimal", "optimal_inaccurate"):
        raise RuntimeError(f"Unexpected solver status: {problem.status}")
    values = np.asarray(x.value, dtype=float)
    if not np.allclose(values, [1, 1], atol=1e-4):
        raise AssertionError("Unexpected optimal production.")
    if not np.isclose(problem.value, 14., atol=1e-4):
        raise AssertionError("Unexpected optimal profit.")
    if not (np.all(A @ values <= b+1e-5) and np.all(values>=-1e-5)):
        raise AssertionError("Solution is infeasible.")
    return values, problem.value, A @ values, b, problem.status

def main():
    values, optimum, lhs, rhs, status = solve_industrial()
    print("Status:", status)
    print("Optimal production [x,y]:", np.round(values, 6))
    print("Maximum profit:", round(float(optimum), 6))
    print("Ax:", np.round(lhs, 6))
    print("b:", rhs)
    print("Constraints satisfied:", np.all(lhs <= rhs + 1e-5))

if __name__ == "__main__":
    main()


The first and third constraints are active at the solution; the second has slack 1. The problem is a convex LP, so the solver's optimal solution is a global maximum. Companion: [Python script](../src/linear_programming.py).